In [1]:
"""
Standardise four monthly tables (CPI, Cocoa, Crude, Exchange Rates)
into ONE common layout so they can be joined safely on `date`.

Common schema for every table
-----------------------------
    date    datetime64[ns]  first day of the month (e.g. 2022-03-01)
    value   float64         the number (NaN when missing)
    unit    string          unit of measurement

HOW TO RUN
----------
Put this .py file in the SAME folder as the four CSV files, then run:

    python transform_tables.py

The cleaned files are written to a sub-folder called  clean_output/
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================================
# 1. FILE NAMES  <-- EDIT HERE if your file names are different
# ============================================================================
# Folder that contains this script (and the CSVs). Works from any location.
BASE_DIR = Path(r"C:\Users\USER\Desktop\academics\pro 1")

CPI_FILE = "CPI Monthly.csv"
COCOA_FILE = "cocoa Monthly.csv"
CRUDE_FILE = "crude Monthly.csv"
EXCHANGE_FILE = "Exchange Rates.csv"

# Folder where the cleaned files will be saved (created automatically)
OUTPUT_DIR = BASE_DIR / "clean_output"

# ============================================================================
COMMON_COLS = ["date", "value", "unit"]

MONTHS = {m: i for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}


# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def parse_period(series: pd.Series) -> pd.Series:
    """'2024M08' -> Timestamp('2024-08-01')."""
    return pd.to_datetime(series.str.strip(), format="%YM%m")


def to_number(series: pd.Series) -> pd.Series:
    """'2,612.80' -> 2612.8 ; '..' / '' / '-' -> NaN."""
    cleaned = (series.astype(str)
               .str.replace(",", "", regex=False)
               .str.strip()
               .replace({"..": np.nan, "": np.nan, "-": np.nan, "nan": np.nan}))
    return pd.to_numeric(cleaned, errors="coerce")


def get_unit(label: str) -> str:
    """'International Brent Crude Oil Price (US$/Barrel) - Monthly Average'
       -> 'US$/Barrel'"""
    m = re.search(r"\(([^)]*)\)", label)
    return m.group(1).strip() if m else ""


def finish(df: pd.DataFrame) -> pd.DataFrame:
    """Force the common layout and types, sort by date."""
    df = df[COMMON_COLS].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = df["value"].astype("float64")
    return df.sort_values("date").reset_index(drop=True)


# ----------------------------------------------------------------------------
# 2. Long files with 'YYYYMmm' periods (CPI, Exchange Rates)
# ----------------------------------------------------------------------------
def read_cpi(path: Path) -> pd.DataFrame:
    # 2 junk lines on top: a title and a blank line
    raw = pd.read_csv(path, skiprows=2, dtype=str)
    raw.columns = ["indicator", "period", "value"]
    raw = raw.dropna(subset=["period"])
    out = pd.DataFrame({
        "date": parse_period(raw["period"]),
        "value": to_number(raw["value"]),
        "unit": "%",
    })
    return finish(out)


def read_exchange_rates(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, skiprows=2, dtype=str)
    period_col, value_col = raw.columns[0], raw.columns[1]
    raw = raw.dropna(subset=[period_col])
    out = pd.DataFrame({
        "date": parse_period(raw[period_col]),
        "value": to_number(raw[value_col]),
        "unit": "GHS per USD",
    })
    # NOTE: 0.00 here is a rounded tiny value (old cedi), NOT missing -> keep it.
    return finish(out)


# ----------------------------------------------------------------------------
# 3. Wide files with Year + Jan..Dec columns (Cocoa, Crude)
# ----------------------------------------------------------------------------
def read_wide_monthly(path: Path, zero_means_missing: bool = True) -> pd.DataFrame:
    raw = pd.read_csv(path, dtype=str)
    long = raw.melt(id_vars=["Year", "Variables"],
                    var_name="month_name", value_name="value")
    long["month"] = long["month_name"].map(MONTHS)
    long["date"] = pd.to_datetime(
        dict(year=long["Year"].astype(int), month=long["month"], day=1))
    long["value"] = to_number(long["value"])

    if zero_means_missing:
        # These files pad future months with 0.00 (prices can't really be 0)
        long.loc[long["value"] == 0, "value"] = np.nan

    long["unit"] = long["Variables"].apply(get_unit)
    return finish(long)


# ----------------------------------------------------------------------------
# 4. Optional: one wide table keyed on date, for modelling / charts
# ----------------------------------------------------------------------------
def make_wide(tables: dict) -> pd.DataFrame:
    """One column per table (named after the table), indexed by date."""
    wide = pd.concat({name: df.set_index("date")["value"]
                      for name, df in tables.items()}, axis=1, sort=True)
    return wide.sort_index()


# ----------------------------------------------------------------------------
# main
# ----------------------------------------------------------------------------
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Each table is transformed independently. Keys become the output file names.
    tables = {
        "cpi": read_cpi(BASE_DIR / CPI_FILE),
        "cocoa": read_wide_monthly(BASE_DIR / COCOA_FILE),
        "crude": read_wide_monthly(BASE_DIR / CRUDE_FILE),
        "exchange rate": read_exchange_rates(BASE_DIR / EXCHANGE_FILE),
    }

    # Export each table individually -> clean_output/<name>_clean.csv
    for name, df in tables.items():
        assert list(df.columns) == COMMON_COLS
        assert str(df["date"].dtype).startswith("datetime64")
        df.to_csv(OUTPUT_DIR / f"{name} clean.csv", index=False)
        print(f"{name:14s} rows={len(df):4d}  "
              f"{df['date'].min():%Y-%m} -> {df['date'].max():%Y-%m}  "
              f"missing={df['value'].isna().sum()}")

    # Optional combined table (one column per series, indexed by date)
    wide = make_wide(tables)
    wide.to_csv(OUTPUT_DIR / "combined.csv")
    print("\nWide table:", wide.shape, "| index dtype:", wide.index.dtype)
    print(wide.dropna(how="all").tail(3))
    print(f"\nFiles saved in: {OUTPUT_DIR}")


